In [2]:
import torch
import torch.nn as nn

print(torch.__version__)

2.8.0


In [28]:
VOCAB_SIZE = 100
EMBEDDING_DIM = 32
HIDDEN_DIM = 64
LAYERS = 2
BATCH_SIZE = 4
SOURCE_LENGTH = 7

In [49]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            input_size = embedding_dim,
            hidden_size = hidden_dim,
            num_layers = num_layers,
            batch_first=True
        )
        
    def forward(
        self,
        src: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]: # type hint
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        
        return outputs, hidden, cell

In [50]:
test_encoder = Encoder(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    LAYERS
)

In [51]:
test_encoder.embedding, test_encoder.lstm

(Embedding(100, 32), LSTM(32, 64, num_layers=2, batch_first=True))

In [52]:
src = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(BATCH_SIZE, SOURCE_LENGTH),
)

outputs, hidden, cell = test_encoder(src)

print("src:", src.shape)
print("outputs:", outputs.shape)
print("hidden:", hidden.shape)
print("cell:", cell.shape)

src: torch.Size([4, 7])
outputs: torch.Size([4, 7, 64])
hidden: torch.Size([2, 4, 64])
cell: torch.Size([2, 4, 64])


In [58]:
'''
src: [4, 7]
4 sequences, each with 7 token IDs

outputs: [4, 7, 64]
For each sequence and each source position, the top LSTM layer exposes a 64-dimensional hidden representation

hidden: [2, 4, 64]
For each of 2 layers and each of 4 sequences, return the final exposed hidden state after all 7 source tokens

cell: [2, 4, 64]
For each of 2 layers and each of 4 sequences, return the final internal cell state after all 7 source tokens
''';

In [54]:
outputs[:, -1, :].shape, hidden[-1].shape, torch.allclose(outputs[:, -1, :], hidden[-1])

(torch.Size([4, 64]), torch.Size([4, 64]), True)

In [59]:
'''
output[:, -1, :] selects the top LSTM layer’s hidden output at the final time step for every batch example:
hidden[-1] selects the final hidden state of the last LSTM layer for every batch example 
''';

In [56]:
for name, parameter in test_encoder.named_parameters():
    print(name, parameter.shape)

embedding.weight torch.Size([100, 32])
lstm.weight_ih_l0 torch.Size([256, 32])
lstm.weight_hh_l0 torch.Size([256, 64])
lstm.bias_ih_l0 torch.Size([256])
lstm.bias_hh_l0 torch.Size([256])
lstm.weight_ih_l1 torch.Size([256, 64])
lstm.weight_hh_l1 torch.Size([256, 64])
lstm.bias_ih_l1 torch.Size([256])
lstm.bias_hh_l1 torch.Size([256])


In [60]:
'''
embedding.weight, 32 dimensions for each of the 100 tokens in vocabulary.
lstm.weight_ih_l0: 256, 32 64 hidden units, 4 gate calculations per unit = 256. Each of the 64 hidden units has 4 gates which are 32 dimensional vectors, which multiply with the 32 input features per token input.
lstm.weight_hh_l0: 256, 64 Each of the 64 neurons need one weight per input gate, forget gate, candidate memory, and output gate, stacked (64 * 4 = 256), takes an input of 64 neuron outputs from the previous hidden state
lstm.weight_ih_l1: 256, 64 The next layer just takes the output of the previous layer, so its 64 neurons output from l0, not the raw 32 dimensions per token.
lstm.weight_hh_l1: 256, 64 The next layer just takes the output of the previous layer, so its 64 neurons output from l0, not the raw 32 dimensions per token.
lstm.bias_ih_l0: 256 just one bias for each of the 4 weights for the LSTM gate * 64 neurons = 256 biases.
lstm.bias_hh_l0: 256 just one bias for each of the 4 weights for the LSTM gate * 64 neurons = 256 biases.
lstm.bias_ih_l1: 256 just one bias for each of the 4 weights for the LSTM gate * 64 neurons = 256 biases.
lstm.bias_hh_l1: 256 just one bias for each of the 4 weights for the LSTM gate * 64 neurons = 256 biases.
''';